In [34]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from utils.metric import metric
from sklearn.model_selection import StratifiedKFold

import lightgbm as lgb

np.random.seed(42)
pd.set_option("display.max_columns", 100)

In [2]:
HORIZONS = [12, 24, 48, 72]

In [3]:
train = pd.read_csv('../../data/train.csv')

In [4]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    eps = 1e-6

    dist = d["dist_min_ci_0_5h"].astype(float).clip(lower=1.0)  # meters
    dist_km = dist / 1000.0

    perims = d["num_perimeters_0_5h"].astype(float).clip(lower=0.0)
    dt_span = d["dt_first_last_0_5h"].astype(float).clip(lower=0.0)

    closing = d["closing_speed_m_per_h"].astype(float)
    closing_pos = closing.clip(lower=0.0)

    radial_rate = d["radial_growth_rate_m_per_h"].astype(float).clip(lower=0.0)
    along_track = d["along_track_speed"].astype(float) if "along_track_speed" in d.columns else 0.0

    align_abs = d["alignment_abs"].astype(float).clip(lower=0.0, upper=1.0)
    align_cos = d["alignment_cos"].astype(float) if "alignment_cos" in d.columns else 0.0

    area = d["area_first_ha"].astype(float).clip(lower=0.0)
    growth_rate_ha_h = d["area_growth_rate_ha_per_h"].astype(float).clip(lower=0.0)

    d["dist_km"] = dist_km
    d["log_dist"] = np.log1p(dist)
    d["sqrt_dist"] = np.sqrt(dist)

    d["inv_dist"] = 1.0 / (dist + 1.0)
    d["inv_dist_sq"] = d["inv_dist"] ** 2
    d["inv_dist_km"] = 1.0 / (dist_km + 0.05)

    d["zone_lt3km"] = (dist < 3000).astype(int)
    d["zone_3to5km"] = ((dist >= 3000) & (dist < 5000)).astype(int)
    d["zone_5to10km"] = ((dist >= 5000) & (dist < 10000)).astype(int)
    d["zone_ge10km"] = (dist >= 10000).astype(int)

    if "dist_fit_r2_0_5h" in d.columns:
        r2 = d["dist_fit_r2_0_5h"].astype(float).clip(lower=0.0, upper=1.0)
        d["dist_trend_reliable"] = (r2 > 0.6).astype(int)
        d["dist_r2"] = r2
    else:
        d["dist_trend_reliable"] = 0
        d["dist_r2"] = 0.0

    for col in ["dist_std_ci_0_5h", "dist_change_ci_0_5h", "dist_slope_ci_0_5h", "dist_accel_m_per_h2"]:
        if col in d.columns:
            d[col] = d[col].astype(float)

    if "dist_change_ci_0_5h" in d.columns:
        d["dist_change_km"] = d["dist_change_ci_0_5h"] / 1000.0
        d["dist_change_norm"] = d["dist_change_ci_0_5h"] / (dist + 1.0)

    if "dist_slope_ci_0_5h" in d.columns:
        d["dist_slope_norm"] = d["dist_slope_ci_0_5h"] / (dist + 1.0)

    if "dist_std_ci_0_5h" in d.columns:
        d["dist_std_norm"] = d["dist_std_ci_0_5h"] / (dist + 1.0)

    effective_closing = closing_pos + radial_rate
    d["effective_closing_speed"] = effective_closing

    d["eta_closing_h"] = np.where(closing_pos > 0.1, dist / (closing_pos + eps), 9999.0)
    d["eta_effective_h"] = np.where(effective_closing > 0.1, dist / (effective_closing + eps), 9999.0)

    d["log_eta_effective"] = np.log1p(np.clip(d["eta_effective_h"], 0, 9999))
    d["log_eta_closing"] = np.log1p(np.clip(d["eta_closing_h"], 0, 9999))

    for h in HORIZONS:
        d[f"eta_within_{h}h"] = (d["eta_effective_h"] <= float(h)).astype(int)
        d[f"eta_margin_{h}h"] = d["eta_effective_h"] - float(h)

    if isinstance(along_track, pd.Series):
        along_pos = along_track.astype(float).clip(lower=0.0)
        d["eta_alongtrack_h"] = np.where(along_pos > 0.1, dist / (along_pos + eps), 9999.0)
        d["log_eta_alongtrack"] = np.log1p(np.clip(d["eta_alongtrack_h"], 0, 9999))
    else:
        d["eta_alongtrack_h"] = 9999.0
        d["log_eta_alongtrack"] = np.log1p(9999.0)

    fire_radius_m = np.sqrt((area * 10000.0) / np.pi)
    d["fire_radius_m"] = fire_radius_m
    d["radius_to_dist"] = fire_radius_m / (dist + 1.0)

    d["dist_minus_radius"] = np.clip(dist - fire_radius_m, 1.0, None)
    d["log_dist_minus_radius"] = np.log1p(d["dist_minus_radius"])

    d["area_to_dist_km"] = area / (dist_km + 0.1)
    d["growth_to_dist_km"] = growth_rate_ha_h / (dist_km + 0.1)

    if "radial_growth_m" in d.columns:
        d["radial_growth_m"] = d["radial_growth_m"].astype(float).clip(lower=0.0)
        d["radial_to_dist"] = d["radial_growth_m"] / (dist + 1.0)

    d["threat_pressure"] = align_abs * effective_closing / (np.log1p(dist) + eps)
    d["alignment_x_closing"] = align_abs * closing_pos
    d["alignment_x_effective"] = align_abs * effective_closing

    if "cross_track_component" in d.columns:
        ctc = d["cross_track_component"].astype(float)
        d["cross_track_abs"] = np.abs(ctc)
        d["cross_track_norm"] = np.abs(ctc) / (dist + 1.0)

    if "spread_bearing_sin" in d.columns and "spread_bearing_cos" in d.columns:
        sb_sin = d["spread_bearing_sin"].astype(float)
        sb_cos = d["spread_bearing_cos"].astype(float)
        d["bearing_strength"] = np.sqrt(sb_sin**2 + sb_cos**2) 

    d["has_movement"] = (perims > 1).astype(int)
    d["perim_density"] = perims / (dt_span + 0.25) 
    d["short_window"] = (dt_span < 0.5).astype(int)

    if "low_temporal_resolution_0_5h" in d.columns:
        d["low_temporal_resolution_0_5h"] = d["low_temporal_resolution_0_5h"].astype(int)

    hour = d["event_start_hour"].astype(float) if "event_start_hour" in d.columns else 0.0
    d["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    d["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)

    dow = d["event_start_dayofweek"].astype(float) if "event_start_dayofweek" in d.columns else 0.0
    d["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    d["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)

    month = d["event_start_month"].astype(float) if "event_start_month" in d.columns else 1.0
    d["month_sin"] = np.sin(2 * np.pi * month / 12.0)
    d["month_cos"] = np.cos(2 * np.pi * month / 12.0)

    d = d.replace([np.inf, -np.inf], np.nan).fillna(0)

    return d

## 24 hour Horizon

In [5]:
def make_strata(t, e):
    strata = np.zeros(len(t), dtype=int)
    
    strata[e == 0] = 0
    
    strata[(e == 1) & (t <= 24)] = 1
    
    strata[(e == 1) & (t > 24) & (t <= 48)] = 2
    
    strata[(e == 1) & (t > 48)] = 3
    
    return strata

In [6]:
train_fe = build_features(train)

In [7]:
def enforce_monotone(P):
    P = np.clip(P, 0.0, 1.0)
    P[:, 1] = np.maximum(P[:, 1], P[:, 0])
    P[:, 2] = np.maximum(P[:, 2], P[:, 1])
    P[:, 3] = np.maximum(P[:, 3], P[:, 2])
    return np.clip(P, 0.0, 1.0)

def masked_brier_for_horizon(p, t, e, H):
    mask = []
    y = []
    for i in range(len(t)):
        if e[i] == 1 and t[i] <= H:
            mask.append(True); y.append(1)
        elif t[i] > H:
            mask.append(True); y.append(0)
        else:
            mask.append(False); y.append(0)
    mask = np.array(mask, dtype=bool)
    y = np.array(y, dtype=int)
    if mask.sum() == 0:
        return 0.0
    return np.mean((p[mask] - y[mask])**2)

In [8]:
drop_cols = ["event_id", "event", "time_to_hit_hours"]

X = train_fe.drop(columns=drop_cols)
t = train_fe["time_to_hit_hours"].values.astype(float)
e = train_fe["event"].values.astype(int)

strata = make_strata(t, e)

X_tr, X_va, t_tr, t_va, e_tr, e_va = train_test_split(
    X, t, e,
    test_size=0.2,
    random_state=42,
    stratify=strata
)


In [9]:
print("Train size:", len(X_tr))
print("Val size:", len(X_va))
print("Train event rate:", e_tr.mean())
print("Val event rate:", e_va.mean())


Train size: 158
Val size: 40
Train event rate: 0.2911392405063291
Val event rate: 0.3


In [10]:
print("Validation distribution:")
print("Val ≤24h:", np.sum((e_va==1)&(t_va<=24)))
print("Val 24-48h:", np.sum((e_va==1)&(t_va>24)&(t_va<=48)))
print("Val 48-72h:", np.sum((e_va==1)&(t_va>48)))
print("Val censored:", np.sum(e_va==0))

Validation distribution:
Val ≤24h: 11
Val 24-48h: 1
Val 48-72h: 0
Val censored: 28


In [11]:
def km_censor_survival(times, censor_event):
    order = np.argsort(times)
    times = times[order]
    censor_event = censor_event[order]

    uniq = np.unique(times)
    n = len(times)

    at_risk = n
    G = 1.0
    G_map = {}

    for tt in uniq:
        m = np.sum(times == tt)
        d = np.sum((times == tt) & (censor_event == 1))

        if at_risk > 0:
            G *= (1.0 - d / at_risk)

        G_map[tt] = max(G, 1e-6)
        at_risk -= m

    uniq_times = np.array(sorted(G_map.keys()))
    G_vals = np.array([G_map[u] for u in uniq_times])
    return uniq_times, G_vals


def G_of_t(t_query, uniq_times, G_vals):
    t_query = np.asarray(t_query)
    idx = np.searchsorted(uniq_times, t_query, side="right") - 1
    out = np.ones_like(t_query, dtype=float)
    ok = idx >= 0
    out[ok] = G_vals[idx[ok]]
    return np.clip(out, 1e-6, 1.0)

In [12]:
c_tr = (e_tr == 0).astype(int)
uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

print("KM built. Example G values:")
print(G_vals[:10])
print("Min G:", G_vals.min())
print("Last 5 G:", G_vals[-5:])

KM built. Example G values:
[1.         1.         1.         1.         1.         1.
 0.99342105 0.99342105 0.99342105 0.99342105]
Min G: 1e-06
Last 5 G: [3.62282390e-02 2.71711792e-02 1.81141195e-02 1.81141195e-02
 1.00000000e-06]


In [13]:
def make_ipcw_data(X, t, e, H, uniq_times, G_vals, weight_cap=30.0):

    y = ((e == 1) & (t <= H)).astype(int)

    mask = ((e == 1) & (t <= H)) | (t > H)

    t_clip = np.minimum(t, H)

    G_t = G_of_t(t_clip, uniq_times, G_vals)

    w = 1.0 / G_t
    w = np.clip(w, 1.0, weight_cap)

    return X[mask], y[mask], w[mask], mask

In [14]:
H = 24

Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
    X_tr, t_tr, e_tr, H, uniq_times, G_vals
)

Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(
    X_va, t_va, e_va, H, uniq_times, G_vals
)

print("Train samples for 24h:", len(Xh_tr))
print("Val samples for 24h:", len(Xh_va))
print("Positive rate (train):", yh_tr.mean())
print("Weight range:", wh_tr.min(), wh_tr.max())

Train samples for 24h: 141
Val samples for 24h: 36
Positive rate (train): 0.2978723404255319
Weight range: 1.0 1.1604062154812953


In [15]:
model_24 = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    max_depth=4,
    min_data_in_leaf=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=8.0,
    objective="binary",
    random_state=42,
)
model_24.fit(Xh_tr, yh_tr)

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Number of positive: 42, number of negative: 99


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007893 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1049
[LightGBM] [Info] Number of data points in the train set: 141, number of used features: 66
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.297872 -> initscore=-0.857450
[LightGBM] [Info] Start training from score -0.857450
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,4
,learning_rate,0.02
,n_estimators,2000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [16]:
p24_val_full = np.zeros(len(X_va))
p24_val_full[mask_va] = model_24.predict_proba(Xh_va)[:, 1]

P_temp = np.zeros((len(X_va), 4))
P_temp[:, 1] = p24_val_full

P_temp = enforce_monotone(P_temp)

c_index, brier, score = metric(P_temp, t_va, e_va)

print("C-index:", c_index)
print("Brier:", brier)
print("Final score:", score)

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
C-index: 0.9315789473684211
Brier: 0.013546001926308684
Final score: 0.9699914828621101


## 12 hour Horizon

In [17]:
print("Full dataset distribution:")
print("≤24h:", np.sum((e==1)&(t<=24)))
print("24–48h:", np.sum((e==1)&(t>24)&(t<=48)))
print("48–72h:", np.sum((e==1)&(t>48)))
print("Censored:", np.sum(e==0))

Full dataset distribution:
≤24h: 53
24–48h: 3
48–72h: 2
Censored: 140


In [18]:
print("Mean predicted 24h prob:", p24_val_full.mean())
print("True 24h rate in val:", np.mean((e_va==1)&(t_va<=24)))

Mean predicted 24h prob: 0.2702771350187487
True 24h rate in val: 0.275


In [19]:
print("≤12h:", np.sum((e==1)&(t<=12)))
print("12–24h:", np.sum((e==1)&(t>12)&(t<=24)))

≤12h: 42
12–24h: 11


In [20]:
H = 12

Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
    X_tr, t_tr, e_tr, H, uniq_times, G_vals
)

Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(
    X_va, t_va, e_va, H, uniq_times, G_vals
)

print("Train samples for 12h:", len(Xh_tr))
print("Val samples for 12h:", len(Xh_va))
print("Positive rate (train):", yh_tr.mean())
print("Weight range:", wh_tr.min(), wh_tr.max())

Train samples for 12h: 154
Val samples for 12h: 38
Positive rate (train): 0.22077922077922077
Weight range: 1.0 1.0295112445358454


In [21]:
model_12 = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    max_depth=4,
    min_data_in_leaf=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=8.0,
    objective="binary",
    random_state=42,
)

model_12.fit(Xh_tr, yh_tr)

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Number of positive: 34, number of negative: 120


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000300 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1157
[LightGBM] [Info] Number of data points in the train set: 154, number of used features: 70
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.220779 -> initscore=-1.261131
[LightGBM] [Info] Start training from score -1.261131
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,4
,learning_rate,0.02
,n_estimators,2000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [22]:
p12_val_full = np.zeros(len(X_va))
p12_val_full[mask_va] = model_12.predict_proba(Xh_va)[:, 1]

P_temp = np.zeros((len(X_va), 4))
P_temp[:, 0] = p12_val_full

P_temp = enforce_monotone(P_temp)

c_index, brier, score = metric(P_temp, t_va, e_va)

print("C-index:", c_index)
print("Brier:", brier)
print("Final score:", score)

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
C-index: 0.9605263157894737
Brier: 0.04814920456176418
Final score: 0.9544534515436072


In [23]:
P_temp = np.zeros((len(X_va), 4))
P_temp[:, 0] = p12_val_full
P_temp[:, 1] = p24_val_full

P_temp = enforce_monotone(P_temp)

c_index, brier, score = metric(P_temp, t_va, e_va)

print("C-index:", c_index)
print("Brier:", brier)
print("Final score:", score)

C-index: 0.9605263157894737
Brier: 0.013017517881506024
Final score: 0.9790456322197878


In [24]:
print("Val 24–48h:", np.sum((e_va==1)&(t_va>24)&(t_va<=48)))

Val 24–48h: 1


In [25]:
print("Mean p12:", P_temp[:,0].mean())
print("Mean p24:", P_temp[:,1].mean())

Mean p12: 0.229843299884907
Mean p24: 0.2730377467282271


## 24 H and 48 H

In [26]:
def make_tail_data(X, t, e, H0, H1, uniq_times, G_vals, weight_cap=50.0):
    y = ((e == 1) & (t > H0) & (t <= H1)).astype(int)

    mask = ((e == 1) & (t > H0) & (t <= H1)) | (t > H1)

    t_clip = np.minimum(t, H1)
    G_t = G_of_t(t_clip, uniq_times, G_vals)
    w = 1.0 / G_t
    w = np.clip(w, 1.0, weight_cap)

    return X[mask], y[mask], w[mask], mask

In [27]:
X48_tr, y48_tr, w48_tr, m48_tr = make_tail_data(X_tr, t_tr, e_tr, 24, 48, uniq_times, G_vals)
X48_va, y48_va, w48_va, m48_va = make_tail_data(X_va, t_va, e_va, 24, 48, uniq_times, G_vals)

print("Tail 24-48 train:", len(X48_tr), "pos:", y48_tr.sum())
print("Tail 24-48 val:", len(X48_va), "pos:", y48_va.sum())

Tail 24-48 train: 76 pos: 2
Tail 24-48 val: 19 pos: 1


In [28]:
tail_feats = [
    "dist_km",
    "eta_effective_h",
    "effective_closing_speed",
    "alignment_abs",
    "has_movement"
]

lr_48 = LogisticRegression(
    C=0.2,
    solver="lbfgs",
    max_iter=2000
)

lr_48.fit(
    X48_tr[tail_feats],
    y48_tr,
    sample_weight=w48_tr
)

p48_tail_full = np.zeros(len(X_va))
p48_tail_full[m48_va] = lr_48.predict_proba(
    X48_va[tail_feats]
)[:, 1]

p48_full = p24_val_full + (1 - p24_val_full) * p48_tail_full

In [29]:
P_temp = np.zeros((len(X_va), 4))
P_temp[:, 0] = p12_val_full
P_temp[:, 1] = p24_val_full
P_temp[:, 2] = p48_full

P_temp = enforce_monotone(P_temp)

c_index, brier, score = metric(P_temp, t_va, e_va)

print("C-index:", c_index)
print("Brier:", brier)
print("Final score:", score)

C-index: 0.9578947368421052
Brier: 0.012254030682675714
Final score: 0.9787905995747586


In [30]:
X72_tr, y72_tr, w72_tr, m72_tr = make_tail_data(X_tr, t_tr, e_tr, 48, 72, uniq_times, G_vals)
X72_va, y72_va, w72_va, m72_va = make_tail_data(X_va, t_va, e_va, 48, 72, uniq_times, G_vals)

print("Tail 48-72 train:", len(X72_tr), "pos:", y72_tr.sum())
print("Tail 48-72 val:", len(X72_va), "pos:", y72_va.sum())

Tail 48-72 train: 2 pos: 2
Tail 48-72 val: 0 pos: 0


In [31]:
late_events = np.sum((e_tr == 1) & (t_tr > 48))
survive_48 = np.sum(t_tr > 48)

alpha = (late_events + 1) / (survive_48 + 2)
p72_full = p48_full + (1 - p48_full) * alpha
print("alpha:", alpha)

alpha: 0.039473684210526314


In [32]:
p72_full = p48_full + (1 - p48_full) * alpha

In [33]:
P_temp = np.zeros((len(X_va), 4))
P_temp[:, 0] = p12_val_full
P_temp[:, 1] = p24_val_full
P_temp[:, 2] = p48_full
P_temp[:, 3] = p72_full

P_temp = enforce_monotone(P_temp)

c_index, brier, score = metric(P_temp, t_va, e_va)

print("C-index:", c_index)
print("Brier:", brier)
print("Final score:", score)

C-index: 0.9578947368421052
Brier: 0.011978292543085002
Final score: 0.9789836162724721
